# 04 - Simulacion Monte Carlo
Simula el Mundial 2026 con el modelo entrenado y un fixture generado a partir de 48 selecciones.

In [1]:
import importlib
import sys
from pathlib import Path

import joblib
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_processed_dataset
from src.features.feature_engineering import build_feature_matrix
from src.simulation import monte_carlo as mc

importlib.reload(mc)

build_groups = mc.build_groups
group_fixtures = mc.group_fixtures
run_monte_carlo_tournament = mc.run_monte_carlo_tournament

MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "xgb.pkl"

In [2]:
# Reemplaza esta lista con las 48 selecciones oficiales
teams = [
    "Argentina", "Brazil", "France", "Germany",
    "Spain", "England", "Portugal", "Netherlands",
    "Italy", "Belgium", "Uruguay", "Colombia",
    "Mexico", "United States", "Canada", "Japan",
    "South Korea", "Australia", "Morocco", "Senegal",
    "Nigeria", "Cameroon", "Ghana", "Egypt",
    "Poland", "Serbia", "Croatia", "Switzerland",
    "Denmark", "Sweden", "Norway", "Austria",
    "Czechia", "Turkey", "Greece", "Ukraine",
    "Chile", "Peru", "Ecuador", "Paraguay",
    "Costa Rica", "Panama", "Saudi Arabia", "Qatar",
    "Iran", "Iraq", "Algeria", "Tunisia",
]

len(teams)

48

In [3]:
# Generar grupos y fixture
import numpy as np

rng_seed = 42
rng = np.random.default_rng(rng_seed)

groups = build_groups(teams, rng)
fixtures = group_fixtures(groups)
fixtures.head()

,group,home_team,away_team
0,A,Peru,Norway
1,A,Peru,Iraq
2,A,Peru,Greece
3,A,Norway,Iraq
4,A,Norway,Greece


In [4]:
# Preparar probabilidades con el modelo entrenado
# Construimos features desde el dataset historico y tomamos el ultimo snapshot por equipo.

import itertools

hist_df = load_processed_dataset()
features_df, feature_cols = build_feature_matrix(hist_df, include_elo=True, apply_decay=True, form_window=5)

# Ultima fila por equipo como local y visitante
home_latest = (
    features_df.sort_values("date")
    .groupby("home_team", as_index=False)
    .tail(1)
    .set_index("home_team")
)
away_latest = (
    features_df.sort_values("date")
    .groupby("away_team", as_index=False)
    .tail(1)
    .set_index("away_team")
)

# Crear todas las combinaciones posibles de partidos (home/away)
pairs = [(h, a) for h, a in itertools.product(teams, teams) if h != a]
proba_base = pd.DataFrame(pairs, columns=["home_team", "away_team"])

# Crear features del fixture a partir de snapshots
fixture_features = proba_base.copy()

fixture_features = fixture_features.join(
    home_latest.add_suffix("_home"), on="home_team"
).join(
    away_latest.add_suffix("_away"), on="away_team"
)

# Reconstruir columnas diff esperadas
fixture_features["rank_diff"] = fixture_features["rank_home_home"] - fixture_features["rank_away_away"]
fixture_features["elo_diff"] = fixture_features["elo_home_home"] - fixture_features["elo_away_away"]
fixture_features["market_value_diff"] = fixture_features["market_value_diff_home"].fillna(0) - fixture_features["market_value_diff_away"].fillna(0)
fixture_features["avg_age_diff"] = fixture_features["avg_age_diff_home"].fillna(0) - fixture_features["avg_age_diff_away"].fillna(0)
fixture_features["squad_size_diff"] = fixture_features["squad_size_diff_home"].fillna(0) - fixture_features["squad_size_diff_away"].fillna(0)
fixture_features["top5_players_diff"] = fixture_features["top5_players_diff_home"].fillna(0) - fixture_features["top5_players_diff_away"].fillna(0)
fixture_features["home_advantage"] = 1
fixture_features["form_win_rate_diff"] = fixture_features["form_win_rate_diff_home"].fillna(0) - fixture_features["form_win_rate_diff_away"].fillna(0)
fixture_features["form_goal_diff_diff"] = fixture_features["form_goal_diff_diff_home"].fillna(0) - fixture_features["form_goal_diff_diff_away"].fillna(0)
fixture_features["form_gf_avg_diff"] = fixture_features["form_gf_avg_diff_home"].fillna(0) - fixture_features["form_gf_avg_diff_away"].fillna(0)
fixture_features["form_ga_avg_diff"] = fixture_features["form_ga_avg_diff_home"].fillna(0) - fixture_features["form_ga_avg_diff_away"].fillna(0)

X_fixture = fixture_features[feature_cols]

model = joblib.load(MODEL_PATH)
proba = model.predict_proba(X_fixture)

proba_df = proba_base.copy()
proba_df["proba_0"] = proba[:, 0]
proba_df["proba_1"] = proba[:, 1]
proba_df["proba_2"] = proba[:, 2]

proba_df.head()

2026-05-24 19:51:29,963 | INFO | src.features.feature_engineering | Adding ELO features
2026-05-24 19:51:30,982 | INFO | src.features.feature_engineering | Adding recent form features
2026-05-24 19:51:31,267 | INFO | src.features.feature_engineering | Adding basic diff features
2026-05-24 19:51:31,277 | INFO | src.features.feature_engineering | Applying time decay


,home_team,away_team,proba_0,proba_1,proba_2
0,Argentina,Brazil,0.081478,0.076901,0.841621
1,Argentina,France,0.093273,0.434500,0.472227
2,Argentina,Germany,0.095619,0.110079,0.794302
3,Argentina,Spain,0.236097,0.115585,0.648318
4,Argentina,England,0.161197,0.060861,0.777942


In [5]:
# Encabezados de la informacion generada
proba_df.columns

Index(['home_team', 'away_team', 'proba_0', 'proba_1', 'proba_2'], dtype='str')

In [6]:
# Ejecutar simulacion Monte Carlo
proba_df["proba_0"] = proba_df["proba_0"].astype(float)
proba_df["proba_1"] = proba_df["proba_1"].astype(float)
proba_df["proba_2"] = proba_df["proba_2"].astype(float)

results = run_monte_carlo_tournament(
    teams=teams,
    proba_df=proba_df,
    n_simulations=100000,
    random_seed=42,
)

# Top-5 selecciones
pd.Series(results).sort_values(ascending=False).head(5)

2026-05-24 19:51:36,475 | INFO | src.simulation.monte_carlo | Running Monte Carlo tournament with 100000 simulations


Spain        14971
Argentina     8803
France        8783
England       8004
Japan         6256
dtype: int64

In [7]:
# Tabla resumen con encabezados claros
simulation_summary = (
    pd.Series(results)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "team", 0: "championships"})
)
simulation_summary["probability"] = simulation_summary["championships"] / simulation_summary["championships"].sum()
simulation_summary.head(10)

,team,championships,probability
0,Spain,14971,0.14971
1,Argentina,8803,0.08803
2,France,8783,0.08783
3,England,8004,0.08004
4,Japan,6256,0.06256
5,Croatia,5904,0.05904
6,Netherlands,3443,0.03443
7,Brazil,2905,0.02905
8,Belgium,2801,0.02801
9,Morocco,2411,0.02411


In [8]:
# Tabla de probabilidades estimadas con encabezados claros
probability_summary = (
    pd.Series(results)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "team", 0: "championships"})
)
probability_summary["estimated_probability"] = probability_summary["championships"] / probability_summary["championships"].sum()
probability_summary[["team", "estimated_probability"]].head(10)

,team,estimated_probability
0,Spain,0.14971
1,Argentina,0.08803
2,France,0.08783
3,England,0.08004
4,Japan,0.06256
5,Croatia,0.05904
6,Netherlands,0.03443
7,Brazil,0.02905
8,Belgium,0.02801
9,Morocco,0.02411


## Notas
- La lista de equipos debe contener exactamente 48 selecciones.
- Si el modelo necesita features adicionales, agrega las columnas requeridas al fixture antes de llamar `predict_probabilities`.